# Causal Mask — Solution

Reference implementation for `02_causal_mask_exercise.ipynb`.

## Setup

In [1]:
import torch

## Solution 1 — `causal_mask`

In [2]:
def causal_mask(seq_len):
    mask = torch.tril(torch.ones((seq_len,seq_len)))
    return mask==1

## Solution 2 — Masked attention

In [3]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_head = Q.shape[-1]
    scores = Q@K.transpose(-2,-1) / d_head**0.5

    if mask is not None:
        scores = scores.masked_fill(~mask,float("-inf"))

    weights = torch.softmax(scores, dim=-1) 
    out = weights@V

    return out

## Solution 3 — Verify the mask works

In [6]:
Q,K,V = torch.chunk(torch.randn(4,8*3),3,dim=-1)
mask = causal_mask(4)

V_modified = V.clone()
V_modified[-1,:] = torch.randn(8)

In [7]:
V

tensor([[ 0.8533, -0.6345,  1.1857, -0.6151, -1.8071,  1.0315,  0.3911, -1.6241],
        [ 1.0885,  0.9144, -0.5796,  2.3575, -1.2692, -0.9810, -0.3217,  0.0260],
        [ 0.3211,  0.7560, -1.3941,  1.3417,  0.9065,  1.2207,  1.0582, -0.0803],
        [-2.0289, -1.0114,  0.5147, -0.3734,  0.0984,  1.3399, -0.9306, -0.7203]])

In [8]:
V_modified

tensor([[ 0.8533, -0.6345,  1.1857, -0.6151, -1.8071,  1.0315,  0.3911, -1.6241],
        [ 1.0885,  0.9144, -0.5796,  2.3575, -1.2692, -0.9810, -0.3217,  0.0260],
        [ 0.3211,  0.7560, -1.3941,  1.3417,  0.9065,  1.2207,  1.0582, -0.0803],
        [ 1.2189, -1.4860, -0.2311,  1.5552,  1.5596,  0.5543, -1.2799,  0.9066]])

In [9]:
output_a = scaled_dot_product_attention(Q, K, V, mask)
output_b = scaled_dot_product_attention(Q, K, V_modified, mask)

In [10]:
for i in range(4):
    if all(output_a[i] == output_b[i]):
        print("Outputs match for positions",i)
    else:
        print("Outputs Dont match for position",i)

Outputs match for positions 0
Outputs match for positions 1
Outputs match for positions 2
Outputs Dont match for position 3


## Run the tests

In [11]:
from tests import run_causal_mask_tests
run_causal_mask_tests(causal_mask, scaled_dot_product_attention)

Running causal mask + masked attention...
  ✓ Mask shape (4, 4)
  ✓ Mask diagonal is True (self-attention)
  ✓ Mask upper triangle is False
  ✓ Mask lower triangle is True
  ✓ Future tokens don't leak past

  All 5 checks passed ✓



True